# UIT DSC 2026 — LegalQA end-to-end v7
Notebook chạy hoàn toàn offline: index corpus, tạo weak labels, checkpoint/resume đầy đủ, fine-tune và sinh `/kaggle/working/submission.zip`. Bật GPU và tắt Internet.

In [ ]:
PROFILE = 'fast'             # fast | balanced | full
TRAIN_QWEN = True
TRAIN_RERANKER = False
QWEN_MAX_SAMPLES = 300       # đặt 300 cho pilot, None cho final
RERANKER_MAX_EXAMPLES = 24000
RERANKER_BATCH_SIZE = 2       # dùng 1 nếu GPU 16–24 GB; 2 cho GPU 40–48 GB
RERANKER_GRAD_ACCUM = 8       # effective batch vẫn là 16
RERANKER_MAX_LENGTH = 512
QWEN_CHECKPOINT_STEPS = 20    # Qwen LoRA checkpoint nhỏ
RERANKER_CHECKPOINT_STEPS = 250 # full-state checkpoint khoảng 6–7 GB
CHECKPOINT_LIMIT = 2
RESUME_TRAINING = True        # tự tìm checkpoint mới nhất
FORCE_INDEX = False
FORCE_TRAIN = False

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
input_root = Path('/kaggle/input')
public_files = sorted(input_root.rglob('public-official.json'))
if len(public_files) != 1:
    raise FileNotFoundError(f'Cần duy nhất một public-official.json: {public_files}')
data_root = public_files[0].parent
train_files = [data_root / 'train.json'] if (data_root / 'train.json').is_file() else []
corpus_dirs = [data_root / 'selected-contexts'] if (data_root / 'selected-contexts').is_dir() else []
if not corpus_dirs:
    context_files = list(input_root.rglob('context_*.json'))
    parent_counts = {}
    for path in context_files:
        parent_counts[path.parent] = parent_counts.get(path.parent, 0) + 1
    corpus_dirs = [max(parent_counts, key=parent_counts.get)] if parent_counts else []
if len(train_files) != 1 or len(corpus_dirs) != 1:
    raise FileNotFoundError(
        f'Không xác định được duy nhất dữ liệu. train={train_files}; public={public_files}; corpus={corpus_dirs}'
    )
def is_legalqa_project(path):
    return (path / 'src' / 'legalqa').is_dir() and (path / 'kaggle' / 'run_end_to_end.py').is_file()
def source_score(path):
    runner = (path / 'kaggle' / 'run_end_to_end.py').read_text(encoding='utf-8', errors='ignore')
    budget = (path / 'src' / 'legalqa' / 'budget.py').read_text(encoding='utf-8', errors='ignore')
    score = 0
    score += 4 if 'data_root = public_path.parent' in runner else 0
    score += 2 if 'public_retrieval.{args.profile}' in runner else 0
    score += 4 if 'unique serialized checkpoint tensors' in budget else 0
    score += 3 if list((path / 'wheelhouse').glob('transformers-5.14.1-*.whl')) else 0
    preflight_ready = (path / 'scripts' / 'preflight_kaggle.py').is_file() and bool(list((path / 'wheelhouse').glob('bitsandbytes-0.49.2-*.whl'))) and bool(list((path / 'wheelhouse').glob('sentencepiece-0.2.1-*.whl')))
    score += 5 if preflight_ready else 0
    score += 4 if '--resume-training' in runner and '--submission-output' in runner else 0
    score -= 5 if (path / 'artifacts' / 'smoke' / 'data' / 'train.json').exists() else 0
    return score
all_source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml') if is_legalqa_project(p.parent)})
if not all_source_projects:
    raise FileNotFoundError('Không tìm thấy LegalQA source project trong /kaggle/input')
source_projects = sorted(all_source_projects, key=lambda path: (source_score(path), str(path)))
selected_source = source_projects[-1]
if source_score(selected_source) < 22:
    raise RuntimeError(f'Source mới chưa đủ bản vá/wheelhouse: {[(str(p), source_score(p)) for p in source_projects]}')
source_dir = Path('/kaggle/working/legalqa_source')
if source_dir.exists():
    shutil.rmtree(source_dir)
shutil.copytree(selected_source, source_dir)
project = source_dir
wheel_dirs = [project / 'wheelhouse']
wheel_dirs.extend(path for path in input_root.rglob('wheelhouse') if path.resolve() != wheel_dirs[0].resolve())
wheel_by_name = {}
for directory in wheel_dirs:
    if directory.is_dir():
        for wheel in sorted(directory.glob('*.whl')):
            wheel_by_name.setdefault(wheel.name, str(wheel))
wheels = sorted(wheel_by_name.values())
required_wheel_prefixes = ('transformers-5.14.1-', 'huggingface_hub-1.26.1-', 'hf_xet-1.6.0-', 'click-8.4.2-', 'tokenizers-0.22.2-', 'regex-2026.7.19-', 'accelerate-1.14.0-', 'safetensors-0.8.0-', 'sentence_transformers-5.7.0-', 'peft-0.18.1-', 'bitsandbytes-0.49.2-', 'sentencepiece-0.2.1-')
missing_wheels = [prefix for prefix in required_wheel_prefixes if not any(Path(wheel).name.startswith(prefix) for wheel in wheels)]
if missing_wheels:
    raise FileNotFoundError(f'Wheelhouse offline chưa đầy đủ: {missing_wheels}')
if wheels:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
    subprocess.run([sys.executable, '-c', "import transformers; from transformers.models.auto.configuration_auto import CONFIG_MAPPING; assert 'qwen3_5' in CONFIG_MAPPING; print('Transformers:', transformers.__version__, '— qwen3_5: OK')"], check=True)
    subprocess.run([sys.executable, '-c', "import torch; from importlib.metadata import version; from packaging.version import Version; bnb_version = version('bitsandbytes'); assert Version(bnb_version) >= Version('0.46.1'); assert torch.cuda.is_available(), 'Kaggle GPU chưa được bật'; print('bitsandbytes:', bnb_version, '— CUDA:', torch.version.cuda, '— GPU:', torch.cuda.get_device_name(0))"], check=True)
src_path = str(project / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Train:', train_files[0])
print('Public:', public_files[0])
print('Corpus:', corpus_dirs[0])
print('Model configs:', len(list(input_root.rglob('config.json'))))
print('Selected source:', selected_source, 'score=', source_score(selected_source))
print('Project:', project)

In [ ]:
command = [
    sys.executable, str(project / 'kaggle' / 'run_end_to_end.py'),
    '--config', str(project / 'configs' / 'kaggle.yaml'),
    '--profile', PROFILE,
    '--input-root', '/kaggle/input',
    '--working', '/kaggle/working/legalqa_run',
    '--train-qwen' if TRAIN_QWEN else '--no-train-qwen',
    '--train-reranker' if TRAIN_RERANKER else '--no-train-reranker',
    '--reranker-max-examples', str(RERANKER_MAX_EXAMPLES),
    '--reranker-batch-size', str(RERANKER_BATCH_SIZE),
    '--reranker-grad-accum', str(RERANKER_GRAD_ACCUM),
    '--reranker-max-length', str(RERANKER_MAX_LENGTH),
    '--reranker-checkpoint-steps', str(RERANKER_CHECKPOINT_STEPS),
    '--qwen-checkpoint-steps', str(QWEN_CHECKPOINT_STEPS),
    '--checkpoint-limit', str(CHECKPOINT_LIMIT),
    '--resume-training' if RESUME_TRAINING else '--no-resume-training',
]
if QWEN_MAX_SAMPLES is not None:
    command += ['--qwen-max-samples', str(QWEN_MAX_SAMPLES)]
if FORCE_INDEX:
    command.append('--force-index')
if FORCE_TRAIN:
    command.append('--force-train')
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, zipfile
submission = Path('/kaggle/working/submission.zip')
assert submission.exists()
with zipfile.ZipFile(submission) as archive:
    assert archive.namelist() == ['submission.json']
    payload = json.loads(archive.read('submission.json').decode('utf-8'))
assert len(payload) == 1000
print(f'READY: {submission} — {submission.stat().st_size:,} bytes — {len(payload)} answers')